# `treasury_curve(maturity, start, end=None, method="flat_forward")`

The US Treasury par yield and zero rate at any maturity from 1M to 30Y, per day, built from the tenors stored in `macro.treasury_rates` (FMP). Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

Each day's curve uses only that day's stored tenors: a missing tenor is never filled, and a maturity past the day's longest tenor gives no row that day. Yields of 1Y and shorter are taken as semiannual zero rates, longer ones as par yields of semiannual-coupon bonds (this module's assumption, not Treasury's published convention). Rates are in percent, compounded semiannually.

In [ ]:
import warnings
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import treasury, treasury_curve
from arora_data.treasury_curve import METHODS, CurvePoint


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The methods
`flat_forward`, the default, bootstraps discount factors tenor by tenor and is linear in log discount factor between them, so the forward rate is flat between tenors. `linear_par` draws straight lines between par yields and bootstraps zero rates from them; it warns, because its forward rates kink at each tenor. More methods (`cubic_par`, `monotone_convex`, `nss`) come in later steps.

In [ ]:
pl.DataFrame({"method": list(METHODS), "warning": [w or "" for _, w in METHODS.values()]})

## One maturity
`maturity="4Y"` over the last week. Maturities are months or years in any case and spacing: `"4Y"`, `"4y"`, `"48M"`, `"2.5Y"`. `lower` and `upper` are the stored tenors either side; at a stored tenor both are that tenor, and its `par_yield` is the stored yield.

In [ ]:
week = today_et() - timedelta(days=7)
assert treasury_curve("4Y", week) == treasury_curve(" 4 y ", week)
show(treasury_curve("4Y", week))

## One day's curve
The latest day at a spread of maturities. Stored tenors come back as stored; par and zero part ways at the long end, where coupons are paid at lower rates than the final one.

In [ ]:
day = treasury(today_et() - timedelta(days=7), tenor="10Y")[-1].date
maturities = ["1M", "3M", "6M", "9M", "1Y", "18M", "2Y", "3Y", "4Y", "5Y", "7Y", "10Y", "15Y", "20Y", "25Y", "30Y"]
points = [treasury_curve(m, day, day)[0] for m in maturities]
stored = {r.tenor: r.yield_pct for r in treasury(day, day)}
curve = to_df(points, CurvePoint).with_columns(stored=pl.Series([stored.get(m) for m in maturities]))
curve.select("maturity", "par_yield", "zero_rate", "stored", "lower", "upper")

## flat_forward against linear_par
The same day. At stored tenors the par yields agree; between them they part, most where tenors are far apart (the 15Y, between 10Y and 20Y, by several basis points). Calling `linear_par` prints its warning, a `MethodWarning`, on every call.

In [ ]:
between = ["9M", "18M", "4Y", "15Y", "25Y"]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    linear = [treasury_curve(m, day, day, "linear_par")[0] for m in between]
flat = [p for p in points if p.maturity in between]
pl.DataFrame({
    "maturity": between,
    "flat_forward par": [p.par_yield for p in flat], "linear_par par": [p.par_yield for p in linear],
    "par gap (bp)": [round((a.par_yield - b.par_yield) * 100, 2) for a, b in zip(flat, linear)],
    "zero gap (bp)": [round((a.zero_rate - b.zero_rate) * 100, 2) for a, b in zip(flat, linear)],
})

In [ ]:
show(treasury_curve("4Y", day, day, "linear_par"))

## Full history
The 4Y, which Treasury does not publish, every day since 1990.

In [ ]:
history = to_df(treasury_curve("4Y", date(1990, 1, 1)), CurvePoint)
history.group_by(pl.col("date").dt.year().alias("year")).agg(
    days=pl.len(), par_min=pl.col("par_yield").min(), par_max=pl.col("par_yield").max(),
    zero_last=pl.col("zero_rate").last()).sort("year")

## Gaps
A maturity past the day's longest tenor is skipped, never extrapolated: with no 30Y stored from 2002-02-19 to 2006-02-08, the 25Y has only the window's two ends. With no 20Y before 1993-10-01, the 15Y comes from the 10Y and 30Y. Before 2001-07-31 the first tenor is 3M, so 1M holds the 3M forward rate flat (`lower` is None).

In [ ]:
show(treasury_curve("25Y", date(2002, 2, 14), date(2006, 2, 10)))

In [ ]:
show(treasury_curve("15Y", date(1992, 6, 1), date(1992, 6, 3)) + treasury_curve("1M", date(1995, 6, 1), date(1995, 6, 1)))

## Bad input
A maturity outside 1M to 30Y, one that is not months or years, or an unknown method raises `ValueError` before the database is read.

In [ ]:
for args in [("31Y", day), ("2W", day), ("10", day), ("4Y", date(2026, 1, 2), date(2026, 1, 1)), ("4Y", day, day, "cubic")]:
    try:
        treasury_curve(*args)
    except ValueError as e:
        print(e)